# RAG Day 1 — 밀 DSS 논문

`PDF → 글자 → 토큰 → 청크 → 임베딩 → 검색 → 답`


모델은 내 문서를 모른다. 답의 모양만 강제하면 모르는 것도 채운다. 처방은 문서를 주는 것이고, 주려면 먼저 글자로 꺼내야 한다. PDF 추출은 코드 세 줄이지만 결과는 도구에 달렸다.


In [ ]:
import sys
from pathlib import Path

_starts = [Path.cwd().resolve()]
_nb = globals().get("__vsc_ipynb_file__")
if isinstance(_nb, str):
    _starts.append(Path(_nb).resolve().parent)
_root = None
for _start in _starts:
    for _p in [_start, *_start.parents]:
        if (_p / "paths.py").is_file() and (_p / "day01").is_dir():
            _root = _p
            break
    if _root is not None:
        break
if _root is None:
    _root = Path.home() / "p3-llm" / "c4-data"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from paths import CORPUS, RAW, TEXT, PDF, MANIFEST, load_api_env

load_api_env()
root = CORPUS
TEXT.mkdir(parents=True, exist_ok=True)
print("CORPUS", CORPUS)
print("PDF.exists", PDF.exists())

import pymupdf
import tiktoken

encoder = tiktoken.get_encoding("o200k_base")


In [ ]:
doc = pymupdf.open(PDF)

text = ""
for page in doc:
    text += page.get_text()

print(len(text))
print(text[:100])


In [ ]:
out_path = TEXT / "paper-wheat-dss.txt"
out_path.write_text(text, encoding="UTF-8")
out_path, out_path.stat().st_size


In [ ]:
pages = [page.get_text() for page in doc]
for i, page in enumerate(doc):
    print(f"{i+1:2d}쪽  {len(page.get_text())}")


In [ ]:
import tiktoken

encoder = tiktoken.get_encoding("o200k_base")
tokens = encoder.encode(text)
print("글자", len(text))
print("토큰", len(tokens))


In [ ]:
import pypdf

reader = pypdf.PdfReader(PDF)
pages_pypdf = [page.extract_text() or "" for page in reader.pages]
print("pymupdf 쪽", len(pages), "글자", sum(len(p) for p in pages))
print("pypdf   쪽", len(pages_pypdf), "글자", sum(len(p) for p in pages_pypdf))


## 쪽별 비교 — 총계만 보면 안 된다

같은 PDF가 추출기에 따라 글자 수가 다르다. 수업이 말한 **3,106자** 차이는 총계다. 어디가 비는지는 **쪽마다** 재야 보인다. 6쪽 표는 오류 없이 조용히 사라지거나 `/cid`로 남는다.


In [ ]:
print(f"{'쪽':>4}  {'pymupdf':>8}  {'pypdf':>8}  {'차이':>8}")
diffs = []
for i, (a, b) in enumerate(zip(pages, pages_pypdf), start=1):
    d = len(b) - len(a)
    diffs.append((abs(d), i, len(a), len(b), d))
    print(f"{i:4d}  {len(a):8d}  {len(b):8d}  {d:+8d}")
print("합계", sum(len(p) for p in pages), sum(len(p) for p in pages_pypdf),
      sum(len(b) - len(a) for a, b in zip(pages, pages_pypdf)))
worst = max(diffs)
print("차이가 큰 쪽:", worst[1], "절댓값", worst[0])


## `repr()` — 화면에 안 그려지는 글자


In [ ]:
i = max(range(len(pages)), key=lambda j: abs(len(pages_pypdf[j]) - len(pages[j])))
print(i + 1, "쪽")
print("--- pymupdf repr ---")
print(repr(pages[i][:400]))
print("--- pypdf repr ---")
print(repr(pages_pypdf[i][:400]))


## `/cid` 세기 · `chr(번호+31)`로 표 되살리기

pypdf가 6쪽에서 더 많이 뽑은 이유는 표 글자가 `/cid56` 같은 토큰으로 남아서다. 이 문서 인코딩은 번호에 31을 더하면 글자가 된다.


In [ ]:
import re

CID = re.compile(r"/cid(\d+)")


def restore_cid(s: str) -> str:
    return CID.sub(lambda m: chr(int(m.group(1)) + 31), s)


for i, (a, b) in enumerate(zip(pages, pages_pypdf), start=1):
    n = len(CID.findall(b))
    if n:
        print(f"{i}쪽  /cid {n}개   pymupdf {len(a)}  pypdf {len(b)}")

restored = restore_cid(pages_pypdf[5])
print("복원 뒤 6쪽 글자", len(restored))
print(restored[restored.find("Table") : restored.find("Table") + 400] if "Table" in restored else restored[800:1200])


## 라이브러리 버전 · 라이선스

내일 장부에 적으려면 지금 확인한다. `importlib.metadata`.


In [ ]:
import importlib.metadata as md

for name in ["pymupdf", "pypdf", "pymupdf4llm", "tiktoken", "beautifulsoup4", "lxml", "markdownify", "pandas"]:
    try:
        meta = md.metadata(name)
        print(f"{name:16} {meta['Version']:10} {meta.get('License', '?')}")
    except md.PackageNotFoundError:
        print(name, "없음")


## pymupdf4llm — 층이 높은 도구

`to_markdown()`은 헤딩을 살린다. `page_chunks=True`는 쪽별 dict. 같은 벽 위에 서서 **얻는 것(구조)**과 **잃는 것(읽기 순서·깨진 글자의 흔적)**이 달라질 뿐이다.


In [ ]:
import pymupdf4llm

md_all = pymupdf4llm.to_markdown(str(PDF))
chunks_md = pymupdf4llm.to_markdown(str(PDF), page_chunks=True)
print("마크다운 글자", len(md_all), "쪽 조각", len(chunks_md))
print("1쪽 keys", list(chunks_md[0].keys()) if chunks_md else None)
print("--- 헤딩이 보이는가 ---")
print("\n".join(line for line in md_all.splitlines() if line.startswith("#"))[:800] or md_all[:400])
print("--- 6쪽 표 자리 ---")
p6 = chunks_md[5]["text"] if len(chunks_md) > 5 else ""
print(p6[:500])


## HTML — 태그를 걷는 순간 구조가 사라진다

위키 「대형 언어 모델」. 본문 셀렉터를 고르고, `markdownify(strip=...)`로 헤딩은 남기고 링크는 버린다. 구조를 뽑는 순간 지키지 않으면 되돌릴 수 없다.


In [ ]:
from bs4 import BeautifulSoup
from markdownify import markdownify as to_md

wiki_html = (RAW / "wiki-llm.html").read_text(encoding="utf-8", errors="ignore")
soup = BeautifulSoup(wiki_html, "lxml")
body = soup.select_one("#mw-content-text")
plain = body.get_text(" ", strip=True) if body else ""
kept = to_md(str(body), heading_style="ATX", strip=["a", "img", "sup"]) if body else ""
print("html 바이트", (RAW / "wiki-llm.html").stat().st_size)
print("태그만 걷은 글자", len(plain), "  헤딩 남긴 글자", len(kept))
print("헤딩 수", sum(1 for line in kept.splitlines() if line.startswith("#")))
print(kept[:400])


## CSV cp949 — 행만 떼면 열 이름이 끊긴다

한국산 CSV는 `utf-8`이 아니라 `cp949`인 경우가 많다. `pd.notna`로 빈 칸을 가르고, 열 이름을 문장에 붙인다.


In [ ]:
import pandas as pd

df = pd.read_csv(RAW / "schedule.csv", encoding="cp949")
print(df.head())
print("행", len(df), "열", list(df.columns))

sentences = []
for _, row in df.iterrows():
    parts = [f"{col}은 {row[col]}" for col in df.columns if pd.notna(row[col])]
    sentences.append("。 ".join(parts) + ".")
print("--- 문장 ---")
print("\n".join(sentences[:5]))


## `build()` — raw/를 훑어 text/와 장부를 세운다

원본은 지우지 않는다. 출처·라이선스·추출기·쪽별 문자수·수집일을 `corpus/manifest.json`에 사람이 읽을 수 있게 적는다. `source`를 `?`로 두면 오늘 과제를 안 한 것이다.


In [ ]:
import json
import shutil
from datetime import date

META = {
    "paper-wheat-dss.pdf": {
        "source": "한국농공학회논문집 66(4) 2024, 김솔희 외. 수업 내 인용.",
        "license": "수업 내 인용",
    },
    "wiki-llm.html": {
        "source": "https://ko.wikipedia.org/wiki/대형_언어_모델",
        "license": "CC BY-SA 4.0",
    },
    "schedule.csv": {
        "source": "생성형 AI 과정 시간표 배포판",
        "license": "과정 내부 배포",
    },
    "guide-day-prev.md": {
        "source": "전일 학습자 가이드 (2026-08-18 구조화 출력과 도구 호출)",
        "license": "수업 자료",
    },
    "pipa.md": {
        "source": "개인정보 보호법 전문 (법제처 / 위키문헌)",
        "license": "저작권법 제7조 자유이용",
    },
}


def extract_pdf(path: Path):
    d = pymupdf.open(path)
    page_chars = [len(p.get_text() or "") for p in d]
    body = "\n\n".join(p.get_text() or "" for p in d)
    return body, page_chars, "pymupdf"


def extract_html(path: Path):
    soup = BeautifulSoup(path.read_text(encoding="utf-8", errors="ignore"), "lxml")
    body = soup.select_one("#mw-content-text") or soup.body or soup
    text = to_md(str(body), heading_style="ATX", strip=["a", "img", "sup"])
    return text, [len(text)], "beautifulsoup4+lxml+markdownify"


def extract_csv(path: Path):
    frame = pd.read_csv(path, encoding="cp949")
    lines = []
    for _, row in frame.iterrows():
        parts = [f"{col}은 {row[col]}" for col in frame.columns if pd.notna(row[col])]
        lines.append("。 ".join(parts) + ".")
    text = "\n".join(lines)
    return text, [len(text)], "pandas/cp949"


def extract_md(path: Path):
    text = path.read_text(encoding="utf-8")
    return text, [len(text)], "plain-utf8"


def extract_one(path: Path):
    suf = path.suffix.lower()
    if suf == ".pdf":
        return extract_pdf(path)
    if suf in {".html", ".htm"}:
        return extract_html(path)
    if suf == ".csv":
        return extract_csv(path)
    if suf in {".md", ".txt"}:
        return extract_md(path)
    raise ValueError(f"모름: {path.name}")


def build():
    TEXT.mkdir(parents=True, exist_ok=True)
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for path in sorted(RAW.iterdir()):
        if not path.is_file() or path.name.startswith("."):
            continue
        body, page_chars, extractor = extract_one(path)
        out = TEXT / (path.stem + ".txt")
        out.write_text(body, encoding="UTF-8")
        info = META.get(path.name, {"source": path.name, "license": "?"})
        rec = {
            "file": path.name,
            "text": out.name,
            "source": info["source"],
            "license": info["license"],
            "extractor": extractor,
            "chars": len(body),
            "page_chars": page_chars,
            "collected": date.today().isoformat(),
        }
        rec["tokens"] = len(encoder.encode(body))
        rows.append(rec)
    MANIFEST.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows


MANIFEST = (root / "corpus" / "manifest.json") if (root / "corpus").is_dir() else (root / "manifest.json")
print("raw", RAW.resolve())
print("text", TEXT.resolve())
print("manifest", MANIFEST.resolve())


In [ ]:
ledger = build()
for rec in ledger:
    print(f"{rec['file']:22} {rec['chars']:7d}자  {rec['tokens']:6d}토큰  {rec['source'][:40]}")
print("합계 글자", sum(r["chars"] for r in ledger), "토큰", sum(r["tokens"] for r in ledger))


## 재현성 — text/를 지워도 장부가 다시 선다

손으로 고친 파일이 `text/`에만 있으면 지우는 순간 사라진다. 원본은 `raw/`에 두고 `build()`만 다시 돌린다.


In [ ]:
before = {p.name: p.read_bytes() for p in TEXT.glob("*.txt")}
shutil.rmtree(TEXT)
ledger2 = build()
after = {p.name: p.read_bytes() for p in TEXT.glob("*.txt")}
print("파일 수", len(before), "→", len(after))
print("바이트 일치", before == after)
print("장부 줄", len(ledger2))


## 코퍼스 토큰 · 임베딩 비용 어림

단가는 기억하지 말고 공식 문서를 본다. `text-embedding-3-small` 입력은 2026-08 기준 **$0.020 / 1M 토큰** (https://platform.openai.com/docs/pricing).


In [ ]:
total_tokens = sum(r["tokens"] for r in ledger2)
usd_per_m = 0.020
cost = total_tokens / 1_000_000 * usd_per_m
print(f"토큰 {total_tokens:,}")
print(f"임베딩 1회 어림 ${cost:.6f}  (단가 ${usd_per_m}/1M, 공식 가격표)")


## 문서를 붙여 근거 있는 답 — 수업 마무리 데모

모델은 내 문서를 모른다. 답의 모양만 강제하면 모르는 것도 채운다. 처방은 **오늘 만든 텍스트를 같이 보내는 것**. 아래는 임베딩 검색이 아니라, 뽑은 글을 그대로 `input`에 붙인다.


In [ ]:
paper_txt = (TEXT / "paper-wheat-dss.txt").read_text(encoding="utf-8")
clip = paper_txt[:6000]
if "client" not in globals():
    from openai import OpenAI
    client = OpenAI()
if "API_MODEL" not in globals():
    API_MODEL = "gpt-5.6-luna"
r = client.responses.create(
    model=API_MODEL,
    input=[
        {"role": "system", "content": "아래 문서에 있는 사실만 답한다. 없으면 문서에 없다고 한다. 근거 문장을 인용한다."},
        {"role": "user", "content": f"문서:\n{clip}\n\n질문: 이 논문이 쓰는 과정기반 작물모형 이름은?"},
    ],
)
print(r.output_text)


## 메이킹 — 내 문서 한 줄

`raw/`에 파일을 하나 더 넣고 `META`에 출처·라이선스를 적은 뒤 `build()`를 다시 돌린다. `source`를 `?`로 남기지 않는다. 스캔 PDF면 `chars: 0`이 장부가 알려 주는 것이다.


## 청크

토큰 400개 단위, 50개 겹침. 쪽 번호는 검색 근거로 남긴다.


In [ ]:
SIZE, OVERLAP = 400, 50
chunks = []
cid = 0
for i, page_text in enumerate(pages, start=1):
    page_tokens = encoder.encode(page_text)
    start = 0
    while start < len(page_tokens):
        piece = page_tokens[start : start + SIZE]
        chunks.append({
            "id": cid,
            "page": i,
            "text": encoder.decode(piece),
        })
        cid += 1
        if start + SIZE >= len(page_tokens):
            break
        start += SIZE - OVERLAP

print("청크", len(chunks))
print(chunks[0]["page"], "쪽 /", chunks[0]["text"][:80])


## 임베딩 · 검색 · 생성


In [ ]:
import os
from openai import OpenAI

client = OpenAI()
API_MODEL = "gpt-5.6-luna"
EMB_MODEL = "text-embedding-3-small"
print("키", "OK" if os.getenv("OPENAI_API_KEY") else "없음")
print(API_MODEL, EMB_MODEL)


In [ ]:
emb = client.embeddings.create(
    model=EMB_MODEL,
    input=[c["text"] for c in chunks],
)
for c, d in zip(chunks, emb.data):
    c["vec"] = d.embedding
print("벡터", len(chunks[0]["vec"]), "조각", len(chunks))


In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb) if na and nb else 0.0


def retrieve(question, k=3):
    qv = client.embeddings.create(model=EMB_MODEL, input=question).data[0].embedding
    ranked = sorted(chunks, key=lambda c: cosine(qv, c["vec"]), reverse=True)
    hits = ranked[:k]
    for h in hits:
        h["score"] = round(cosine(qv, h["vec"]), 3)
    return hits


def ask(question, k=3):
    hits = retrieve(question, k=k)
    context = "\n\n".join(f"[{h['page']}쪽]\n{h['text']}" for h in hits)
    r = client.responses.create(
        model=API_MODEL,
        input=[
            {
                "role": "system",
                "content": (
                    "논문 조각만 근거로 답한다. 없으면 '논문에 없다'고 한다. "
                    "답 끝에 쪽수를 적는다."
                ),
            },
            {"role": "user", "content": f"자료:\n{context}\n\n질문: {question}"},
        ],
    )
    print("질문:", question)
    print("근거:", [(h["page"], h["score"]) for h in hits])
    print(r.output_text)
    print()
    return r.output_text


In [ ]:
ask("이 시스템이 사용하는 과정기반 작물모형은 무엇인가?")
ask("시험에 쓴 밀 품종과 포장 위치는?")
ask("2023년 수확기 기준 최대·최소 잠재생산량(kg/ha)과 누적강수량은?")
ask("밀 생육단계를 몇 단계로 나눴고, 기준은 무엇인가?")
ask("이 시스템으로 사과 과원의 적과 일정을 짜 줄 수 있나?")
